In [ ]:
pip install pandas numpy scikit-learn shap streamlit joblib nltk

  Using cached shap-0.49.1-cp310-cp310-win_amd64.whl (547 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.15.0
    Uninstalling typing-extensions-4.15.0:
      Successfully uninstalled typing-extensions-4.15.0
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\enith\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:
import pandas as pd

df = pd.read_csv("fake_job_postings.csv", on_bad_lines='skip', engine='python')

print("Original shape:", df.shape)
print("Missing values per column:\n", df.isnull().sum())

# Fill missing TEXT fields with empty string (safe for NLP later)
text_cols = ["title", "location", "company_profile", "description",
             "requirements", "benefits", "employment_type",
             "required_experience", "required_education", "industry", "function"]

for col in text_cols:
    df[col] = df[col].fillna("")

# Fill missing binary/flag fields with 0
flag_cols = ["telecommuting", "has_company_logo", "has_questions"]
for col in flag_cols:
    df[col] = df[col].fillna(0)

# Drop rows where the label itself is missing (unusable for training)
df = df.dropna(subset=["fraudulent"])
df["fraudulent"] = df["fraudulent"].astype(int)

# Combine all text fields into ONE column — makes text processing easier downstream
df["full_text"] = (
    df["title"] + " " + df["company_profile"] + " " +
    df["description"] + " " + df["requirements"] + " " + df["benefits"]
)

print("\nClean shape:", df.shape)
print("Fake vs Real counts:\n", df["fraudulent"].value_counts())

# Save clean file for the rest of the team
df.to_csv("clean_job_postings.csv", index=False)
print("\nSaved: clean_job_postings.csv")

Original shape: (17880, 18)
Missing values per column:
 job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64

Clean shape: (17880, 19)
Fake vs Real counts:
 fraudulent
0    17014
1      866
Name: count, dtype: int64

Saved: clean_job_postings.csv


In [ ]:
import pandas as pd
import re
df = pd.read_csv("clean_job_postings.csv")
df["full_text"] = df["full_text"].fillna("")

# --- Clue 1: Urgency language ---
urgency_words = ["apply now", "urgent", "immediate start", "limited time",
                  "act fast", "hurry", "don't miss"]

def count_urgency(text):
    text = text.lower()
    return sum(text.count(word) for word in urgency_words)

df["urgency_score"] = df["full_text"].apply(count_urgency)

# --- Clue 2: Vagueness score (count how many key fields are missing/blank) ---
def vagueness_score(row):
    score = 0
    if row["company_profile"].strip() == "":
        score += 1
    if row["required_education"].strip() == "":
        score += 1
    if row["industry"].strip() == "":
        score += 1
    if row["benefits"].strip() == "":
        score += 1
    return score

df["vagueness_score"] = df.apply(vagueness_score, axis=1)

# --- Clue 3: Text length ---
df["text_length"] = df["full_text"].str.split().str.len()

# --- Clue 4: Exclamation / all-caps intensity ---
def caps_and_punctuation_score(text):
    exclamations = text.count("!")
    caps_words = len(re.findall(r"\b[A-Z]{3,}\b", text))
    return exclamations + caps_words

df["hype_score"] =(
     df["full_text"].str.count("!") + 
     df["full_text"].str.findall(r"\b[A-Z]{3,}\b").str.len())

# Keep the structured flags from the raw data too
final_cols = ["full_text", "urgency_score", "vagueness_score", "text_length",
              "hype_score", "telecommuting", "has_company_logo",
              "has_questions", "fraudulent"]

df_final = df[final_cols]
df_final.to_csv("featured_job_postings.csv", index=False)
print("Saved: featured_job_postings.csv")
print(df_final.head())